# Swarm Walk-Forward Analysis Template

**Zero setup. Run all cells.**

Walk-forward analysis is the gold standard for backtesting. It prevents overfitting by strictly separating train and test windows, then rolling forward through time.

This template supports:
- **Anchored** (expanding train window) and **Rolling** (fixed train window)
- Any asset available on Yahoo Finance
- Custom strategy injection (just implement `generate_signal`)
- Full performance reporting: Sharpe, drawdown, hit rate, equity curves
- Pre-registration support: declare predictions before running

Connects to: PDD-002 (Compressed Research Cycle Validation)

In [ ]:
# Cell 0: Setup (only external dependency: yfinance)
!pip install -q yfinance

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

print('Ready.')

In [ ]:
# Cell 1: Configuration — EDIT THIS

@dataclass
class WalkForwardConfig:
    # Assets
    tickers: list = field(default_factory=lambda: ['SPY', 'QQQ', 'GLD', 'TLT'])

    # Walk-forward windows (trading days)
    train_days: int = 252        # 1 year training
    test_days: int = 63          # 3 months testing
    step_days: int = 21          # 1 month step

    # Date range
    start_date: str = '2018-01-01'
    end_date: str = '2026-03-31'

    # Mode: 'anchored' (expanding window) or 'rolling' (fixed window)
    mode: str = 'anchored'

    # Transaction costs (one-way, fraction)
    cost_bps: float = 5.0  # 5 basis points

cfg = WalkForwardConfig()
print(f'Config: {cfg.mode} | train={cfg.train_days}d test={cfg.test_days}d step={cfg.step_days}d')
print(f'Assets: {cfg.tickers}')
print(f'Period: {cfg.start_date} to {cfg.end_date}')

In [ ]:
# Cell 2: Pre-registration — STATE YOUR PREDICTIONS BEFORE RUNNING
# This cell is your scientific notary. Fill it in, then don't touch it.

PRE_REGISTRATION = {
    'filed_at': datetime.now().isoformat(),
    'hypothesis': 'Momentum signal (12-1 month) generates positive risk-adjusted returns OOS',
    'predicted_sharpe': 0.4,          # your prediction
    'predicted_hit_rate': 0.53,       # your prediction
    'predicted_max_drawdown': -0.15,  # your prediction (negative)
    'falsified_if': 'OOS Sharpe < 0.0 (strategy is not better than cash)',
    'notes': 'Simple 12-1 momentum, long-only, equal weight across assets',
}

print('=== PRE-REGISTRATION ===')
for k, v in PRE_REGISTRATION.items():
    print(f'  {k}: {v}')
print('========================')
print('DO NOT EDIT THIS CELL AFTER RUNNING BACKTEST')

In [ ]:
# Cell 3: Data download

def fetch_data(tickers, start, end):
    """Download adjusted close prices from Yahoo Finance."""
    data = yf.download(tickers, start=start, end=end, auto_adjust=True)['Close']
    if isinstance(data, pd.Series):
        data = data.to_frame(tickers[0])
    data = data.dropna()
    print(f'Downloaded {len(data)} trading days, {len(data.columns)} assets')
    print(f'Date range: {data.index[0].date()} to {data.index[-1].date()}')
    return data

prices = fetch_data(cfg.tickers, cfg.start_date, cfg.end_date)
returns = prices.pct_change().dropna()
prices.tail()

In [ ]:
# Cell 4: Strategy — REPLACE THIS WITH YOUR OWN LOGIC

def generate_signal(train_prices: pd.DataFrame, train_returns: pd.DataFrame) -> pd.Series:
    """
    Given training data, produce a signal for each asset.

    Returns:
        pd.Series indexed by ticker, values are weights (positive = long).
        Weights are normalized to sum to 1 (fully invested).

    This default implementation: 12-1 month momentum.
    Buy assets with positive 12m return, skip most recent month.
    Equal-weight the winners.
    """
    if len(train_prices) < 252:
        # Not enough data, equal weight everything
        n = len(train_prices.columns)
        return pd.Series(1.0 / n, index=train_prices.columns)

    # 12-month return, skip last month (12-1 momentum)
    mom_start = train_prices.iloc[-252]
    mom_end = train_prices.iloc[-21]
    momentum = (mom_end - mom_start) / mom_start

    # Long assets with positive momentum, equal weight
    winners = momentum[momentum > 0]
    if len(winners) == 0:
        # All negative momentum -> go to cash (zero weight)
        return pd.Series(0.0, index=train_prices.columns)

    weights = pd.Series(0.0, index=train_prices.columns)
    weights[winners.index] = 1.0 / len(winners)
    return weights

In [ ]:
# Cell 5: Walk-Forward Engine (DO NOT EDIT unless you know what you're doing)

@dataclass
class WFWindow:
    train_start: int
    train_end: int
    test_start: int
    test_end: int
    weights: Optional[pd.Series] = None

def build_windows(n_obs: int, cfg: WalkForwardConfig) -> list:
    """Generate train/test window indices."""
    windows = []
    i = 0
    while True:
        if cfg.mode == 'anchored':
            tr_start = 0
        else:
            tr_start = i * cfg.step_days

        tr_end = cfg.train_days + i * cfg.step_days
        te_start = tr_end
        te_end = te_start + cfg.test_days

        if te_end > n_obs:
            break

        windows.append(WFWindow(tr_start, tr_end, te_start, te_end))
        i += 1

    return windows

def run_walkforward(prices: pd.DataFrame, returns: pd.DataFrame, cfg: WalkForwardConfig):
    """Execute the walk-forward analysis."""
    windows = build_windows(len(prices), cfg)
    print(f'Generated {len(windows)} walk-forward windows')

    cost = cfg.cost_bps / 10000
    all_oos_returns = []
    prev_weights = None

    for w in windows:
        # Split
        train_p = prices.iloc[w.train_start:w.train_end]
        train_r = returns.iloc[w.train_start:w.train_end]
        test_r = returns.iloc[w.test_start:w.test_end]

        # Generate signal from training data only (NO LOOK-AHEAD)
        weights = generate_signal(train_p, train_r)
        w.weights = weights

        # Transaction costs on rebalance
        if prev_weights is not None:
            turnover = (weights - prev_weights).abs().sum()
        else:
            turnover = weights.abs().sum()
        rebal_cost = turnover * cost

        # Out-of-sample portfolio returns
        port_ret = (test_r * weights).sum(axis=1)
        # Apply transaction cost on first day of test period
        port_ret.iloc[0] -= rebal_cost

        all_oos_returns.append(port_ret)
        prev_weights = weights

    oos = pd.concat(all_oos_returns)
    # Remove duplicate indices (overlapping windows)
    oos = oos[~oos.index.duplicated(keep='first')].sort_index()

    return oos, windows

oos_returns, windows = run_walkforward(prices, returns, cfg)
print(f'Out-of-sample period: {oos_returns.index[0].date()} to {oos_returns.index[-1].date()}')
print(f'Total OOS trading days: {len(oos_returns)}')

In [ ]:
# Cell 6: Performance metrics

def calc_metrics(oos: pd.Series, annual_factor: int = 252) -> dict:
    """Calculate standard performance metrics."""
    total_ret = (1 + oos).prod() - 1
    n_years = len(oos) / annual_factor
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
    ann_vol = oos.std() * np.sqrt(annual_factor)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0

    # Drawdown
    cum = (1 + oos).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()

    # Hit rate
    hit_rate = (oos > 0).mean()

    # Calmar ratio
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0

    return {
        'Total Return': f'{total_ret:.2%}',
        'Annual Return': f'{ann_ret:.2%}',
        'Annual Volatility': f'{ann_vol:.2%}',
        'Sharpe Ratio': f'{sharpe:.3f}',
        'Max Drawdown': f'{max_dd:.2%}',
        'Calmar Ratio': f'{calmar:.3f}',
        'Hit Rate (daily)': f'{hit_rate:.2%}',
        'Trading Days': len(oos),
        'Years': f'{n_years:.1f}',
    }

metrics = calc_metrics(oos_returns)
print('\n=== OUT-OF-SAMPLE PERFORMANCE ===')
for k, v in metrics.items():
    print(f'  {k:25s} {v}')
print('=================================')

In [ ]:
# Cell 7: Visualization

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Equity curve
cum = (1 + oos_returns).cumprod()
axes[0].plot(cum.index, cum.values, 'b-', linewidth=1.2)
axes[0].set_ylabel('Growth of $1')
axes[0].set_title('Walk-Forward Equity Curve (Out-of-Sample Only)')
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Drawdown
running_max = cum.cummax()
dd = (cum - running_max) / running_max
axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.3)
axes[1].plot(dd.index, dd.values, 'r-', linewidth=0.8)
axes[1].set_ylabel('Drawdown')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

# Rolling Sharpe (63-day)
roll_ret = oos_returns.rolling(63).mean() * 252
roll_vol = oos_returns.rolling(63).std() * np.sqrt(252)
roll_sharpe = roll_ret / roll_vol
axes[2].plot(roll_sharpe.index, roll_sharpe.values, 'g-', linewidth=1.0)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Rolling Sharpe (63d)')
axes[2].set_title('Rolling Sharpe Ratio')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Walk-forward window visualization

fig, ax = plt.subplots(figsize=(14, max(4, len(windows) * 0.25)))

for i, w in enumerate(windows):
    train_dates = prices.index[w.train_start:w.train_end]
    test_dates = prices.index[w.test_start:w.test_end]

    ax.barh(i, (train_dates[-1] - train_dates[0]).days,
            left=train_dates[0], height=0.6, color='steelblue', alpha=0.7)
    ax.barh(i, (test_dates[-1] - test_dates[0]).days,
            left=test_dates[0], height=0.6, color='coral', alpha=0.7)

ax.set_ylabel('Window #')
ax.set_title(f'Walk-Forward Windows ({cfg.mode})')
ax.legend(['Train', 'Test'], loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nWeight evolution across windows:')
for i, w in enumerate(windows[-5:]):
    if w.weights is not None:
        active = w.weights[w.weights > 0]
        if len(active) > 0:
            print(f'  Window {len(windows)-5+i}: {dict(active.round(3))}')
        else:
            print(f'  Window {len(windows)-5+i}: CASH (no positive momentum)')

In [ ]:
# Cell 9: Calibration check — compare results to pre-registration

def score_preregistration(metrics_dict, prereg):
    """Compare actual results against pre-registered predictions."""
    sharpe_actual = float(metrics_dict['Sharpe Ratio'])
    hit_actual = float(metrics_dict['Hit Rate (daily)'].strip('%')) / 100
    dd_actual = float(metrics_dict['Max Drawdown'].strip('%')) / 100

    print('\n=== CALIBRATION SCORECARD ===')
    print(f'{"Metric":25s} {"Predicted":>12s} {"Actual":>12s} {"Error":>12s} {"Status":>10s}')
    print('-' * 75)

    checks = [
        ('Sharpe Ratio', prereg['predicted_sharpe'], sharpe_actual),
        ('Hit Rate', prereg['predicted_hit_rate'], hit_actual),
        ('Max Drawdown', prereg['predicted_max_drawdown'], dd_actual),
    ]

    errors = []
    for name, pred, actual in checks:
        if pred != 0:
            rel_err = abs(actual - pred) / abs(pred)
        else:
            rel_err = abs(actual - pred)
        errors.append(rel_err)
        status = 'GOOD' if rel_err < 0.3 else 'MISS'
        print(f'{name:25s} {pred:12.3f} {actual:12.3f} {rel_err:11.1%} {status:>10s}')

    mean_err = np.mean(errors)
    print(f'\nMean relative error: {mean_err:.1%}')
    print(f'Calibration grade: {"WELL-CALIBRATED" if mean_err < 0.3 else "NEEDS WORK"}')

    # Falsification check
    print(f'\nFalsification test: {prereg["falsified_if"]}')
    if sharpe_actual < 0:
        print('RESULT: FALSIFIED. Strategy does not beat cash OOS.')
    else:
        print(f'RESULT: NOT FALSIFIED (Sharpe = {sharpe_actual:.3f} > 0)')

    return mean_err

calibration_error = score_preregistration(metrics, PRE_REGISTRATION)

In [ ]:
# Cell 10: Adversarial review — TRY TO DESTROY YOUR OWN FINDING
# Run this AFTER you see the results. Be honest.

def adversarial_review(oos: pd.Series, windows: list, cfg: WalkForwardConfig):
    """Automated checks for common self-deception patterns."""
    print('=== ADVERSARIAL REVIEW ===')
    issues = []

    # 1. Regime dependence: does strategy only work in one regime?
    mid = len(oos) // 2
    first_half = oos.iloc[:mid]
    second_half = oos.iloc[mid:]
    s1 = first_half.mean() / first_half.std() * np.sqrt(252) if first_half.std() > 0 else 0
    s2 = second_half.mean() / second_half.std() * np.sqrt(252) if second_half.std() > 0 else 0
    if s1 * s2 < 0:
        issues.append(f'REGIME DEPENDENCE: Sharpe flips sign (1st half: {s1:.2f}, 2nd half: {s2:.2f})')
    else:
        print(f'  [PASS] Regime consistency: 1st half Sharpe={s1:.2f}, 2nd half={s2:.2f}')

    # 2. Small sample: enough OOS data?
    if len(oos) < 504:
        issues.append(f'SMALL SAMPLE: Only {len(oos)} OOS days (<2 years). Results may not generalize.')
    else:
        print(f'  [PASS] Sample size: {len(oos)} OOS days')

    # 3. Concentration: is all return from a few days?
    sorted_ret = oos.sort_values(ascending=False)
    top10_contribution = sorted_ret.iloc[:10].sum() / oos.sum() if oos.sum() != 0 else 0
    if top10_contribution > 0.5:
        issues.append(f'CONCENTRATION: Top 10 days = {top10_contribution:.0%} of total return. Fragile.')
    else:
        print(f'  [PASS] Return distribution: top 10 days = {top10_contribution:.0%} of total')

    # 4. Survivorship: are we only testing liquid assets that survived?
    issues.append('MANUAL CHECK: Are tested assets survivorship-biased? (ETFs less prone, single stocks more)')

    # 5. Cost sensitivity
    no_cost_ret = oos.copy()
    actual_sharpe = oos.mean() / oos.std() * np.sqrt(252) if oos.std() > 0 else 0
    doubled_cost = oos - (cfg.cost_bps / 10000 * 2 / 252)  # rough doubled-cost approximation
    cost_sharpe = doubled_cost.mean() / doubled_cost.std() * np.sqrt(252) if doubled_cost.std() > 0 else 0
    if cost_sharpe < 0:
        issues.append(f'COST SENSITIVE: Doubling costs kills the strategy (Sharpe: {actual_sharpe:.2f} -> {cost_sharpe:.2f})')
    else:
        print(f'  [PASS] Cost robustness: 2x costs -> Sharpe {cost_sharpe:.2f} (still positive)')

    # 6. Parameter sensitivity: would slightly different params give opposite results?
    print(f'  [MANUAL] Parameter sensitivity: re-run with train={cfg.train_days-42}d and train={cfg.train_days+42}d')

    print(f'\n--- ISSUES FOUND: {len(issues)} ---')
    for issue in issues:
        print(f'  [!] {issue}')

    verdict = 'PROCEED WITH CAUTION' if len([i for i in issues if 'MANUAL' not in i]) <= 1 else 'INVESTIGATE FURTHER'
    print(f'\nVerdict: {verdict}')
    return issues

issues = adversarial_review(oos_returns, windows, cfg)

In [ ]:
# Cell 11: Comparison — run both anchored and rolling, compare

results = {}
for mode in ['anchored', 'rolling']:
    c = WalkForwardConfig(mode=mode, tickers=cfg.tickers,
                          start_date=cfg.start_date, end_date=cfg.end_date)
    oos, _ = run_walkforward(prices, returns, c)
    m = calc_metrics(oos)
    results[mode] = m

print('\n=== ANCHORED vs ROLLING ===')
print(f'{"Metric":25s} {"Anchored":>12s} {"Rolling":>12s}')
print('-' * 52)
for key in results['anchored']:
    print(f'{key:25s} {str(results["anchored"][key]):>12s} {str(results["rolling"][key]):>12s}')

In [ ]:
# Cell 12: Export results for PDD-002 scoring

result_record = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'mode': cfg.mode,
        'tickers': cfg.tickers,
        'train_days': cfg.train_days,
        'test_days': cfg.test_days,
        'period': f'{cfg.start_date} to {cfg.end_date}',
    },
    'pre_registration': PRE_REGISTRATION,
    'metrics': metrics,
    'calibration_error': float(calibration_error),
    'adversarial_issues': len(issues),
    'n_windows': len(windows),
}

# Print as copy-pasteable record
import json
print(json.dumps(result_record, indent=2, default=str))
print('\nCopy this into your prediction log.')

---

## How to use this template

### Quick start
1. Open in Colab (File > Open notebook > GitHub or Upload)
2. Edit **Cell 1** (config) and **Cell 2** (pre-registration)
3. Edit **Cell 4** (`generate_signal`) with your strategy
4. Run all cells
5. Read the calibration scorecard and adversarial review honestly

### Swap in your own strategy
Replace `generate_signal` in Cell 4. It receives training data and returns a weight Series. That's it. Everything else (walk-forward, metrics, visualization) stays the same.

### Strategy ideas to test
- **Mean reversion**: short-term reversal (5-day) instead of momentum
- **Volatility targeting**: scale weights by inverse realized vol
- **Risk parity**: weight by inverse variance
- **ML signal**: train a simple model (sklearn) on features, predict direction

### Rules for credible research
1. Fill in pre-registration BEFORE seeing results
2. DO NOT edit pre-registration after running
3. Run adversarial review and take it seriously
4. If falsified, publish the failure — it builds more credibility than hiding it
5. Git-commit the notebook with results for timestamped proof